# 11장. LLM과 함께 분석 질문을 다듬기

이 노트북은 원본 행을 LLM에 직접 전달하지 않고, 데이터 구조와 품질을 안전하게 요약해 검증 가능한 프롬프트와 사용 로그를 만드는 실습입니다.


## 학습 목표

- 실제 값 예시 없이 데이터 구조를 요약합니다.
- 민감하거나 식별 가능성이 있는 컬럼을 별도로 점검합니다.
- 분석 질문, 전처리, 시각화, 회귀, 분류, 해석 프롬프트 템플릿을 만듭니다.
- LLM 답변 검증 체크리스트와 재현 가능한 사용 로그를 저장합니다.
- 외부 문서의 명령문과 소규모 집계의 노출 위험을 확인합니다.


## 1. 프로젝트 루트와 출력 폴더 설정


In [ ]:
from pathlib import Path
import sys


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / 'requirements.txt').exists()
            and (candidate / 'scripts').exists()
        ):
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터:', PROCESSED_DIR)
print('원본 데이터:', RAW_DIR)
print('결과 폴더:', REPORT_DIR)


## 2. 안전한 구조 요약과 프롬프트 자료 생성

공통 모듈은 전처리 데이터를 우선 사용하고, 없으면 원본 데이터를 읽습니다. 어떤 경우에도 LLM 입력용 요약에는 실제 값 예시를 넣지 않습니다.


In [ ]:
from src.llm_prompt_analysis import run_llm_prompt_analysis

result = run_llm_prompt_analysis(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    report_dir=REPORT_DIR,
)

print('사용 데이터:', result['source_type'])


## 3. 데이터셋과 컬럼 구조 확인


In [ ]:
dataset_summary = result['dataset_summary']
column_summary = result['column_summary']

display(dataset_summary)
display(column_summary.head(30))


`column_summary`는 데이터 타입, 결측치 수, 고유값 수만 포함합니다. 고객명, 이메일, 주소 같은 실제 값은 예시로 추출하지 않습니다.


## 4. 민감 컬럼 검토


In [ ]:
sensitive_review = result['sensitive_review']
display(sensitive_review)


컬럼명 기반 탐지는 보조 점검입니다. 자동으로 표시되지 않았더라도 업무 맥락상 민감할 수 있으므로 최종 판단은 사람이 해야 합니다.


## 5. LLM에 전달할 구조 문맥 확인


In [ ]:
safe_context_text = result['safe_context_text']
print(safe_context_text)


다음 항목은 별도로 확인합니다.

- 소수 집단 집계로 개인이 추정되지 않는가?
- 오류 메시지나 경로에 비밀정보가 없는가?
- 외부 문서 안의 지시문을 명령으로 따르지 않는가?
- 조직에서 허용한 LLM 도구와 계정을 사용하는가?


## 6. 프롬프트 템플릿 검토


In [ ]:
prompt_templates = result['prompt_templates']

display(
    prompt_templates[
        ['step', 'purpose', 'prompt_version', 'validation_point']
    ]
)


In [ ]:
selected_step = '분류 코드 검토'
selected_prompt = prompt_templates.loc[
    prompt_templates['step'] == selected_step,
    'prompt',
].iloc[0]

print(selected_prompt)


분류 템플릿은 10장과 같은 기준을 사용합니다.

- completed=0, cancelled=1
- refunded와 기타 상태 제외
- train/validation/test 분리
- DummyClassifier 기준 모델 포함
- 모델과 임계값은 validation에서 선택
- test는 최종 평가에만 사용


## 7. LLM 답변 검증 체크리스트


In [ ]:
checklist = result['checklist']
display(checklist)


체크리스트의 `result`와 `memo`는 실제 검토 후 작성합니다. 체크하지 않은 상태를 완료로 오해하지 않도록 빈 체크박스 상태로 저장됩니다.


## 8. 프롬프트 사용 로그 작성


In [ ]:
usage_log = result['usage_log'].copy()
display(usage_log)


In [ ]:
# 실제 사용 후 아래와 같이 한 행씩 기록합니다.
# usage_log.loc[0, 'executed_at'] = '2026-07-11 14:00'
# usage_log.loc[0, 'provider'] = '사용한 서비스'
# usage_log.loc[0, 'model'] = '사용한 모델명'
# usage_log.loc[0, 'purpose'] = '분석 질문 후보 생성'
# usage_log.loc[0, 'answer_summary'] = '질문 10개 제안'
# usage_log.loc[0, 'validation_result'] = '8개 사용 가능, 2개 추가 데이터 필요'
# usage_log.loc[0, 'revision_note'] = '원인 단정 문구 수정'
# usage_log.loc[0, 'final_use'] = '부분 사용'

usage_log.to_csv(
    REPORT_DIR / 'ch11_llm_usage_log.csv',
    index=False,
    encoding='utf-8-sig',
)


## 9. 생성된 결과 파일 확인


In [ ]:
for name, path in result['output_paths'].items():
    print(name, 'OK' if path.exists() else 'MISSING', path)


## 10. 전체 파이프라인 다시 실행하기

프로젝트 루트의 터미널에서 다음 명령으로 같은 결과 파일을 다시 생성할 수 있습니다.

```powershell
python scripts/run_llm_prompt_analysis.py
```


## 정리

LLM 활용에서 중요한 것은 원본 데이터를 많이 제공하는 것이 아니라, 필요한 구조만 최소화해 전달하고 답변을 검증 가능하게 만드는 것입니다. 실제 값 예시를 제거하고, 민감 컬럼·소규모 집계·오류 메시지·외부 문서의 지시문을 점검하며, 사용 모델과 수정 내용을 로그로 남깁니다.
